# Part A — Few-Shot Classification with Transfer Learning

**Group 7 | Office-Home | Art → Clipart | Office Products**

Classes: Calculator, Keyboard, Laptop, Monitor, Mouse, Printer

Three strategies are compared:
1. **From scratch** — ResNet-50 with random weights (lower-bound baseline)
2. **Feature extraction (frozen)** — pretrained backbone frozen, only head trained
3. **Fine-tuning** — layer3 + layer4 + head unfrozen, low LR for backbone

Each strategy is run with 3 independent seeds. Results are reported as mean ± std.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.utils import get_device, set_seed, build_domain_splits, make_loaders, SEEDS, CLASSES
from src.classifier import (
    build_resnet_scratch, build_resnet_frozen, build_resnet_finetuned,
    count_trainable_params, train_model, run_multi_seed_experiment
)

device = get_device()
print(f'Device: {device}')

## 1. Data Loading — Art (source) and Clipart (target)

In [ ]:
# Split source domain (Art): 70% train, 15% val, 15% test
# Note: Art has 18-51 images/class — we use ALL available images with stratified splits
art_train, art_val, art_test = build_domain_splits('Art', seed=SEEDS[0])
clip_train, clip_val, clip_test = build_domain_splits('Clipart', seed=SEEDS[0])

print('Art   — train:', len(art_train), 'val:', len(art_val), 'test:', len(art_test))
print('Clipart — train:', len(clip_train), 'val:', len(clip_val), 'test:', len(clip_test))

# Build loaders (fixed test sets — same split across seeds)
_, _, src_test_loader  = make_loaders(art_train, art_val, art_test)
_, _, tgt_test_loader  = make_loaders(clip_train, clip_val, clip_test)

## 2. Count trainable parameters

In [ ]:
params = {
    'scratch':   count_trainable_params(build_resnet_scratch()),
    'frozen':    count_trainable_params(build_resnet_frozen()),
    'finetuned': count_trainable_params(build_resnet_finetuned()),
}
for name, n in params.items():
    print(f'{name:12s}: {n:,} trainable parameters')

## 3. Multi-seed training and evaluation

In [ ]:
# Factory functions that rebuild loaders per seed (ensures reproducibility)
def make_train_loader(seed):
    tr, va, _ = build_domain_splits('Art', seed=seed)
    loader, _, _ = make_loaders(tr, va, [])
    return loader

def make_val_loader(seed):
    tr, va, _ = build_domain_splits('Art', seed=seed)
    _, loader, _ = make_loaders(tr, va, [])
    return loader

results = {}

# Strategy 1: From scratch
results['scratch'] = run_multi_seed_experiment(
    'scratch', make_train_loader, make_val_loader,
    src_test_loader, tgt_test_loader,
    num_epochs=30,
    checkpoint_dir='../checkpoints',
)

# Strategy 2: Frozen backbone
results['frozen'] = run_multi_seed_experiment(
    'frozen', make_train_loader, make_val_loader,
    src_test_loader, tgt_test_loader,
    num_epochs=30,
    checkpoint_dir='../checkpoints',
)

# Strategy 3: Fine-tuning
results['finetuned'] = run_multi_seed_experiment(
    'finetuned', make_train_loader, make_val_loader,
    src_test_loader, tgt_test_loader,
    num_epochs=50,
    checkpoint_dir='../checkpoints',
)

## 4. Summary table (Δ_shift)

In [ ]:
rows = []
for strategy, r in results.items():
    rows.append({
        'Strategy': strategy,
        'Src Acc': f"{r['src_mean']:.4f} ± {r['src_std']:.4f}",
        'Tgt Acc': f"{r['tgt_mean']:.4f} ± {r['tgt_std']:.4f}",
        'Δ_shift': f"{r['delta_shift_mean']:.4f}",
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 5. Figure A — Training curves (loss and accuracy)

In [ ]:
# Re-run a single seed to capture history for plotting
set_seed(SEEDS[0])
tr, va, _ = build_domain_splits('Art', seed=SEEDS[0])
train_loader, val_loader, _ = make_loaders(tr, va, [])

histories = {}
for strategy, builder in [('scratch', build_resnet_scratch), ('frozen', build_resnet_frozen), ('finetuned', build_resnet_finetuned)]:
    set_seed(SEEDS[0])
    model = builder()
    res = train_model(model, train_loader, val_loader, strategy=strategy, num_epochs=30)
    histories[strategy] = res['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'scratch': 'tab:red', 'frozen': 'tab:blue', 'finetuned': 'tab:green'}

for strategy, h in histories.items():
    epochs = range(1, len(h['train_loss']) + 1)
    axes[0].plot(epochs, h['train_loss'], '--', color=colors[strategy], alpha=0.6, label=f'{strategy} (train)')
    axes[0].plot(epochs, h['val_loss'],   '-',  color=colors[strategy], label=f'{strategy} (val)')
    axes[1].plot(epochs, h['train_acc'],  '--', color=colors[strategy], alpha=0.6)
    axes[1].plot(epochs, h['val_acc'],    '-',  color=colors[strategy])

axes[0].set(title='Loss curves', xlabel='Epoch', ylabel='Cross-Entropy Loss')
axes[1].set(title='Accuracy curves', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/part_a_training_curves.png', dpi=150)
plt.show()
print('Saved → figures/part_a_training_curves.png')

## 6. Figure D (preview) — t-SNE of backbone features (before adaptation)

In [ ]:
from sklearn.manifold import TSNE
import torch.utils.data as data_utils

# Load the best finetuned model (seed=42)
best_model = build_resnet_finetuned()
ckpt_path = '../checkpoints/finetuned_seed42.pt'
best_model.load_state_dict(torch.load(ckpt_path, map_location=device))
best_model = best_model.to(device)

# Extract penultimate features (avgpool output) — remove fc
import torchvision.models as models
import torch.nn as nn

class FeatureExtractorWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        # ResNet-50 forward without fc
        x = self.model.conv1(x); x = self.model.bn1(x); x = self.model.relu(x)
        x = self.model.maxpool(x)
        x = self.model.layer1(x); x = self.model.layer2(x)
        x = self.model.layer3(x); x = self.model.layer4(x)
        x = self.model.avgpool(x)
        return torch.flatten(x, 1)

extractor = FeatureExtractorWrapper(best_model).to(device)
extractor.eval()

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            f = extractor(imgs.to(device)).cpu().numpy()
            feats.append(f); labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)

src_feats, src_labels = extract_features(src_test_loader)
tgt_feats, tgt_labels = extract_features(tgt_test_loader)

all_feats  = np.concatenate([src_feats, tgt_feats])
all_labels = np.concatenate([src_labels, tgt_labels])
all_domain = np.array([0] * len(src_feats) + [1] * len(tgt_feats))

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb  = tsne.fit_transform(all_feats)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_cls = plt.cm.tab10(np.linspace(0, 1, len(CLASSES)))

# By class
for i, cls in enumerate(CLASSES):
    mask = all_labels == i
    axes[0].scatter(emb[mask, 0], emb[mask, 1], c=[colors_cls[i]], label=cls, alpha=0.6, s=15)
axes[0].set_title('t-SNE coloured by class (before adaptation)')
axes[0].legend(fontsize=7, markerscale=2)

# By domain
for dom, name, color in [(0, 'Art (source)', 'steelblue'), (1, 'Clipart (target)', 'coral')]:
    mask = all_domain == dom
    axes[1].scatter(emb[mask, 0], emb[mask, 1], c=color, label=name, alpha=0.5, s=15)
axes[1].set_title('t-SNE coloured by domain (before adaptation)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/tsne_before_adaptation.png', dpi=150)
plt.show()
print('Saved → figures/tsne_before_adaptation.png')